In [ ]:
!pip install -q langchain langchain-groq gspread

In [ ]:
import os
from google.colab import userdata, auth

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

print("Groq connected successfully! ")

Groq connected successfully! 


In [ ]:
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

sh = gc.create("Sweet Crumbs Orders")
ws = sh.sheet1

ws.append_row(["name", "item", "qty", "date", "eggless", "total"])

print("Google Sheet created successfully!")
print("Sheet link:", sh.url)


Google Sheet created successfully!
Sheet link: https://docs.google.com/spreadsheets/d/1OGJC2HaG7FyunsNAuWnCDQgenAZxdFQs_oHHNs3LDwQ


In [ ]:
MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

HOURS = {
    "monday": "9 AM to 8 PM",
    "tuesday": "9 AM to 8 PM",
    "wednesday": "9 AM to 8 PM",
    "thursday": "9 AM to 8 PM",
    "friday": "9 AM to 8 PM",
    "saturday": "9 AM to 8 PM",
    "sunday": "10 AM to 6 PM"
}

print("Menu and working hours created successfully! ")

Menu and working hours created successfully! 


In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class Order(BaseModel):
    """Customer order details."""

    name: str = Field(description="Customer's name")

    item: Literal[
        "plum cake",
        "chocolate cake",
        "cupcake"
    ] = Field(description="Selected bakery item")

    qty: int = Field(
        description="Number of items",
        ge=1
    )

    date: str = Field(description="Pickup date")

    eggless: bool = Field(
        description="Whether the customer wants an eggless item"
    )

print("Order model created successfully! ")

Order model created successfully! 


In [ ]:
from langchain.tools import tool

@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price for a bakery item.

    Args:
        item: Name of the bakery item.
        qty: Number of items required.
    """
    item = item.lower().strip()

    if item not in MENU:
        return f"Sorry, {item} is not available in our menu."

    if qty < 1:
        return "Quantity must be at least 1."

    total = MENU[item] * qty

    return f"{qty} x {item} = Rs. {total}"

print("get_price tool created successfully! ")

get_price tool created successfully! 


In [ ]:
@tool
def is_open(day: str) -> str:
    """Check the bakery's opening hours for a given day.

    Args:
        day: Day of the week, such as Monday or Sunday.
    """
    day = day.lower().strip()

    if day not in HOURS:
        return "Please provide a valid day of the week."

    return f"We are open on {day}: {HOURS[day]}."

print("is_open tool created successfully! ")

is_open tool created successfully! 


In [ ]:
@tool
def log_order(
    name: str,
    item: str,
    qty: int,
    date: str,
    eggless: bool
) -> str:
    """Save a confirmed customer order to Google Sheets.

    Args:
        name: Customer's name.
        item: Bakery item.
        qty: Quantity required.
        date: Pickup date.
        eggless: True if eggless is requested.
    """
    try:
        order = Order(
            name=name,
            item=item.lower().strip(),
            qty=qty,
            date=date,
            eggless=eggless
        )
    except Exception as e:
        return f"Invalid order: {e}"

    total = MENU[order.item] * order.qty

    ws.append_row([
        order.name,
        order.item,
        order.qty,
        order.date,
        "Yes" if order.eggless else "No",
        total
    ])

    return f"Order saved! Total amount: Rs. {total}"

print("log_order tool created successfully! ")

log_order tool created successfully! 


In [ ]:
from langchain.agents import create_agent

SYSTEM = """
You are a friendly customer service assistant for Sweet Crumbs Home Bakery.

Rules:
1. Use get_price to check all item prices.
2. Use is_open to check working hours.
3. Ask for the customer's name, item, quantity and pickup date.
4. Never save an order if any required information is missing.
5. Never accept zero or negative quantities.
6. Use log_order only after collecting all required details.
7. Confirm the order and total amount after saving.
8. Be polite and keep answers short.
"""

agent = create_agent(
    model=llm,
    tools=[get_price, is_open, log_order],
    system_prompt=SYSTEM
)

print("Sweet Crumbs AI Agent created successfully! ")

Sweet Crumbs AI Agent created successfully! 


In [ ]:
history = []

def chat(message):
    global history

    result = agent.invoke({
        "messages": history + [
            {"role": "user", "content": message}
        ]
    })

    history = result["messages"]

    return history[-1].content

print(chat("Hi! What cakes are available?"))

Sure! We offer a variety of cakes—just let me know what you’d like. Could you share your name, the cake you’re interested in, how many you’d like, and your pickup date? If you need an egg‑less option, just let me know!


In [ ]:
history = []

print(chat("Are you open on Sunday?"))
print(chat("What is the price of 2 chocolate cakes?"))

Yes! We’re open on Sunday from 10 AM to 6 PM. Let me know if you’d like to place an order.
Two chocolate cakes cost Rs. 1,300 in total. Let me know your name, the quantity you’d like, and a pickup date, and I’ll log the order for you.


In [ ]:
history = []

print(chat("Do you sell donuts?"))

Yes, we do!  
Could you please tell me your name, the item you’d like, the quantity, and the pickup date? (Let me know if you’d like an egg‑less option.)


In [ ]:
SYSTEM = """
You are a friendly customer service assistant for Sweet Crumbs Home Bakery.

Rules:
1. Our ONLY menu items are plum cake, chocolate cake and cupcake.
2. Never say an item is available unless it exists in MENU.
3. If an item is not in MENU, politely say it is unavailable.
4. Always use get_price to check prices.
5. Always use is_open to check working hours.
6. Collect customer name, item, quantity and pickup date.
7. Never accept zero or negative quantities.
8. Never save incomplete or invalid orders.
9. Use log_order only after collecting all required details.
10. Confirm the order and total after saving.
"""

agent = create_agent(
    model=llm,
    tools=[get_price, is_open, log_order],
    system_prompt=SYSTEM
)

history = []

print(chat("Do you sell donuts?"))

I’m sorry, but we don’t offer donuts. We specialize in plum cake, chocolate cake, and cupcakes. Let me know if you’d like to order one of those instead!


In [ ]:
history = []

print(chat("I'm Priya. I want 2 cupcakes. Please book my order."))

Hi Priya! Great choice.  
Could you let me know the pickup date you’d like? (e.g., “Friday, 12 Nov”)


In [ ]:
history = []

print(chat("I'm Priya. I want 0 cupcakes for 24 December."))

Hi Priya! I see you’d like to order cupcakes for 24 December. Unfortunately, we can’t process an order with a quantity of 0. Could you let me know how many cupcakes you’d like? Once we have a positive quantity, I’ll check our opening hours for that day and confirm the order for you.


In [ ]:
for message in history:
    message.pretty_print()

================================ Human Message =================================

I'm Priya. I want 0 cupcakes for 24 December.
================================== Ai Message ==================================

Hi Priya! I see you’d like to order cupcakes for 24 December. Unfortunately, we can’t process an order with a quantity of 0. Could you let me know how many cupcakes you’d like? Once we have a positive quantity, I’ll check our opening hours for that day and confirm the order for you.


In [ ]:
print(ws.get_all_records())

[]


In [ ]:
history = []

print(chat(
    "I'm Ravi. I want 2 chocolate cakes for 24 December. "
    "Please make them eggless and save my order."
))

Sure thing, Ravi! To make sure we’re open on the day you’d like to pick up, could you let me know which day of the week 24 December falls on? Once I have that, I’ll confirm the order and total for you.


In [ ]:
print(chat("24 December 2026 is Thursday. Please proceed with my order."))

All set! Your order for **2 eggless chocolate cakes** is saved for pickup on **24 December 2026 (Thursday)**. The total comes to **Rs. 1300**. We look forward to seeing you then—happy baking!


In [ ]:
print(ws.get_all_records())

[{'name': 'Ravi', 'item': 'chocolate cake', 'qty': 2, 'date': '24 December 2026', 'eggless': 'Yes', 'total': 1300}]
